# nb_04 — DimProduct

**Purpose:** Load `DimProduct` from `Files/seed_data/dim_product.csv` with SCD Type 2 scaffold and a surrogate key.

## Overview
- Reads `dim_product.csv`
- Adds surrogate key `product_key` via `monotonically_increasing_id()`
- Adds SCD2 columns: `scd_start_date`, `scd_end_date`, `is_current`
- Writes to Delta table `DimProduct`

**Prerequisite:** nb_00_setup_lakehouse must be run first.

In [ ]:
from pyspark.sql.functions import monotonically_increasing_id, current_date, lit

# Read seed CSV
df = spark.read.csv(
    "Files/seed_data/dim_product.csv",
    header=True,
    inferSchema=True
)

# Add surrogate key and SCD2 scaffold
df = (
    df
    .withColumn("product_key", monotonically_increasing_id())
    .withColumn("scd_start_date", current_date())
    .withColumn("scd_end_date", lit("9999-12-31").cast("date"))
    .withColumn("is_current", lit(True))
)

print(f"Rows read: {df.count()}")
df.printSchema()

In [ ]:
# Write to Delta table
(
    df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("DimProduct")
)

print("DimProduct loaded successfully.")
spark.sql("SELECT COUNT(*) AS row_count FROM DimProduct").show()
spark.sql("SELECT product_key, is_current, scd_start_date, scd_end_date FROM DimProduct LIMIT 5").show()

In [ ]:
# SCD2 upsert helper for incremental product updates
from delta.tables import DeltaTable

def upsert_scd2(spark, table, source_df, nk, tracked_cols):
    dt = DeltaTable.forName(spark, table)
    (
        dt.alias("t")
        .merge(
            source_df.alias("s"),
            f"t.{nk} = s.{nk} AND t.is_current = true"
        )
        .whenMatchedUpdate(
            condition=" OR ".join(f"s.{c} != t.{c}" for c in tracked_cols),
            set={"scd_end_date": "current_date()", "is_current": "false"}
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

print("upsert_scd2 helper defined.")